# 12 — Train/Test với dữ liệu synthetic 3.000 rows

So sánh `real train`, `enriched 3000` và `synthetic-only 3000` trên cùng một `locked real test`. Test synthetic không được dùng để tuyên bố chất lượng lâm sàng. Notebook không xuất model.

In [ ]:
!pip install -q gdown scikit-learn xgboost lightgbm

In [ ]:
import time
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from xgboost import XGBClassifier

RANDOM_STATE = 42
EXPERIMENT_SEEDS = [42, 52, 62, 72, 82]
DOWNLOAD_DIR = Path('/content/heart_data_drive')
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 100)

## 1. Tải và tự nhận diện 5 file Drive

Notebook dùng file ID, tải với tên tạm rồi nhận diện theo số dòng/cột và `data_origin`. Tất cả link phải được chia sẻ ở chế độ *Anyone with the link — Viewer*.

In [ ]:
DRIVE_FILES = {
    'file_1':'1YTzUy_RreXqnM5fqMR0hOLZPeXOvYoju',
    'file_2':'1ezKHfVWmu5kQHBKQllfrr5YEBU4wQJpn',
    'file_3':'1SOaF95H5QoUi0OkvuLdeAwFskG-x772L',
    'file_4':'170nGcuSSvitwMgepyEYDjHgpQdAyZ42Q',
    'file_5':'1sWTkNCcXE3lVcKZmfbjYE9g7gCqIh1da',
}
downloaded = []
for alias, file_id in DRIVE_FILES.items():
    path = DOWNLOAD_DIR / alias
    result = gdown.download(id=file_id, output=str(path), quiet=False, fuzzy=True)
    if result is None:
        raise RuntimeError(f'Khong tai duoc {alias}; hay kiem tra quyen chia se Drive')
    downloaded.append(Path(result))
print('Downloaded:', [str(p) for p in downloaded])

In [ ]:
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = [c for c in FEATURES if c not in NUMERICAL_FEATURES]
ALL_COLUMNS = FEATURES + ['target']

datasets, manifest_text = {}, None
for path in downloaded:
    raw = path.read_bytes()
    try:
        candidate = pd.read_csv(path)
    except Exception:
        manifest_text = raw.decode('utf-8', errors='replace')
        continue
    if set(ALL_COLUMNS).issubset(candidate.columns):
        if len(candidate) == 3000 and 'data_origin' in candidate.columns:
            origins = set(candidate['data_origin'].astype(str).unique())
            if 'real_train' in origins:
                datasets['enriched_train'] = candidate
            else:
                datasets['synthetic_demo'] = candidate
        elif len(candidate) < 100:
            datasets['locked_test'] = candidate
        continue
    # Cleveland goc khong co header.
    try:
        original = pd.read_csv(path, header=None, names=FEATURES+['target_original'], na_values=['?'])
        if len(original) == 303:
            original['target'] = (pd.to_numeric(original['target_original'], errors='coerce') > 0).astype(int)
            datasets['original'] = original[ALL_COLUMNS]
    except Exception:
        manifest_text = raw.decode('utf-8', errors='replace')

required = {'original','enriched_train','synthetic_demo','locked_test'}
missing = required - set(datasets)
if missing:
    raise ValueError(f'Khong nhan dien duoc: {sorted(missing)}')
for name, data in datasets.items():
    print(name, data.shape, 'positive_rate=', round(float(data['target'].mean()),4))
if manifest_text:
    print('Manifest:', manifest_text[:500])

## 2. Kiểm tra tính toàn vẹn và dựng real train

Real train được lấy trực tiếp từ các dòng `data_origin=real_train` trong enriched dataset. Kiểm tra test không xuất hiện trong real train hoặc synthetic demo dưới dạng exact match.

In [ ]:
original_df = datasets['original'][ALL_COLUMNS].copy()
enriched_df = datasets['enriched_train'].copy()
synthetic_df = datasets['synthetic_demo'].copy()
locked_test_df = datasets['locked_test'][ALL_COLUMNS].copy()
real_train_df = enriched_df[enriched_df['data_origin']=='real_train'][ALL_COLUMNS].copy()

for frame in [original_df, enriched_df, synthetic_df, locked_test_df, real_train_df]:
    for column in ALL_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors='coerce')

def exact_overlap(left, right):
    return left[ALL_COLUMNS].merge(right[ALL_COLUMNS].drop_duplicates(), how='inner').shape[0]

integrity = {
    'original_rows':len(original_df), 'real_train_rows':len(real_train_df),
    'enriched_rows':len(enriched_df), 'synthetic_rows':len(synthetic_df),
    'locked_test_rows':len(locked_test_df),
    'real_train_test_overlap':exact_overlap(real_train_df, locked_test_df),
    'synthetic_test_exact_overlap':exact_overlap(synthetic_df, locked_test_df),
}
display(pd.Series(integrity, name='integrity'))
assert len(original_df)==303 and len(enriched_df)==3000 and len(synthetic_df)==3000
assert integrity['real_train_test_overlap']==0, 'Data leakage: real train trung locked test'

training_sets = {
    'real_only':real_train_df,
    'enriched_3000':enriched_df[ALL_COLUMNS],
    'synthetic_only_3000':synthetic_df[ALL_COLUMNS],
}
display(pd.DataFrame([{'train_set':n,'rows':len(d),'positive_rate':d.target.mean(),
                       'missing':int(d[FEATURES].isna().sum().sum())}
                      for n,d in training_sets.items()]).round(4))

## 3. Benchmark nhiều model và seed

Mọi cấu hình đánh giá trên cùng locked real test. `synthetic_only_3000` là phép đo Train-on-Synthetic, Test-on-Real (TSTR), không phải đề xuất mặc định.

In [ ]:
def make_preprocessor():
    numeric = Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True)),
                        ('scaler',MinMaxScaler())])
    categorical = Pipeline([('imputer',SimpleImputer(strategy='most_frequent',add_indicator=True)),
                            ('encoder',OneHotEncoder(handle_unknown='ignore'))])
    return ColumnTransformer([('num',numeric,NUMERICAL_FEATURES),
                              ('cat',categorical,CATEGORICAL_FEATURES)])

def make_models(seed):
    return {
      'Logistic Regression':LogisticRegression(max_iter=1500,random_state=seed),
      'Random Forest':RandomForestClassifier(n_estimators=400,min_samples_leaf=2,random_state=seed,n_jobs=-1),
      'Extra Trees':ExtraTreesClassifier(n_estimators=400,min_samples_leaf=2,random_state=seed,n_jobs=-1),
      'XGBoost':XGBClassifier(n_estimators=300,max_depth=3,learning_rate=.03,subsample=.8,
          colsample_bytree=.8,eval_metric='logloss',random_state=seed,n_jobs=-1),
      'LightGBM':LGBMClassifier(n_estimators=300,num_leaves=15,learning_rate=.03,
          random_state=seed,verbosity=-1,n_jobs=-1)}

records = []
for train_name, training in training_sets.items():
    for seed in EXPERIMENT_SEEDS:
        for model_name, classifier in make_models(seed).items():
            pipeline = Pipeline([('preprocessor',make_preprocessor()),('classifier',classifier)])
            started = time.perf_counter()
            pipeline.fit(training[FEATURES],training['target'].astype(int))
            fit_seconds = time.perf_counter()-started
            probability = pipeline.predict_proba(locked_test_df[FEATURES])[:,1]
            prediction = (probability >= .5).astype(int)
            tn,fp,fn,tp = confusion_matrix(locked_test_df['target'],prediction,labels=[0,1]).ravel()
            records.append({'train_set':train_name,'model':model_name,'seed':seed,
                'accuracy':accuracy_score(locked_test_df['target'],prediction),
                'precision':precision_score(locked_test_df['target'],prediction,zero_division=0),
                'recall':recall_score(locked_test_df['target'],prediction,zero_division=0),
                'specificity':tn/(tn+fp) if tn+fp else np.nan,
                'false_negative_rate':fn/(fn+tp) if fn+tp else np.nan,
                'f1':f1_score(locked_test_df['target'],prediction,zero_division=0),
                'roc_auc':roc_auc_score(locked_test_df['target'],probability),
                'brier':brier_score_loss(locked_test_df['target'],probability),
                'false_negatives':int(fn),'fit_seconds':fit_seconds})
results = pd.DataFrame(records)
display(results.head())

In [ ]:
summary = results.groupby(['train_set','model']).agg(
    accuracy_mean=('accuracy','mean'),accuracy_std=('accuracy','std'),
    precision_mean=('precision','mean'),recall_mean=('recall','mean'),
    recall_std=('recall','std'),recall_min=('recall','min'),
    f1_mean=('f1','mean'),roc_auc_mean=('roc_auc','mean'),roc_auc_std=('roc_auc','std'),
    brier_mean=('brier','mean'),false_negatives_mean=('false_negatives','mean'),
    fit_seconds_mean=('fit_seconds','mean')).reset_index().sort_values(
        ['roc_auc_mean','recall_mean','brier_mean'],ascending=[False,False,True])
display(summary.round(4))

plt.figure(figsize=(14,6))
sns.barplot(data=summary,x='model',y='roc_auc_mean',hue='train_set')
plt.ylim(.5,1); plt.xticks(rotation=25); plt.title('Locked real test ROC AUC'); plt.show()

winner = summary.iloc[0]
print('EMPIRICAL WINNER ON LOCKED REAL TEST')
display(winner)

## 4. Cách kết luận

- Nếu `enriched_3000` thắng real-only: kiểm tra thêm repeated data splits; một locked test khoảng 61 dòng chưa đủ chốt.
- Nếu synthetic-only gần real-only: synthetic có downstream utility tốt, nhưng vẫn không thay thế real data.
- Ưu tiên ROC-AUC, Recall, false negatives và Brier; không chọn chỉ theo Accuracy.
- Mọi kết luận phải ghi rõ 3.000 rows được sinh từ 242 real train patients.
- Sau khi chọn candidate, bước tiếp theo là external validation trên dữ liệu bệnh viện thật.